# Tutorial 2: Forward Prediction with ARIA

**ARIA: Causal-Aware Reasoning for Materials Discovery**

---

## Introduction

Forward prediction asks: **given synthesis conditions, what properties will result?** ARIA answers this through a **3-tier reasoning cascade** that gates evidence activation on causal completeness:

| Tier | Name | Trigger | Evidence | Confidence |
|------|------|---------|----------|------------|
| 1 | Direct | Exact PSP path found in KG | Complete causal chain | Highest |
| 2 | Analogical | Similar material found via embeddings | Transferred causal chain | Medium |
| 3 | Fallback | No KG match | Pure LLM parametric knowledge | Lowest |

The critical insight: **naive retrieval (just dumping KG edges into the prompt) can be worse than no retrieval at all** if the retrieved evidence is causally incomplete. ARIA's tier system ensures that evidence is only activated when it forms a complete PSP chain.

### What You Will Learn

1. Initializing `ARIAEngine` in different operating modes
2. Running forward predictions and inspecting `ARIAResult`
3. Understanding tier routing decisions
4. Comparing modes: baseline vs naive_kg vs aria
5. Visualizing causal traces with PSP layer coloring

---

## 1. Setup

In [ ]:
import networkx as nx
import json
import os

# ARIA imports
from aria.types import (
    ARIAResult,
    CausalTraceStep,
    EngineMode,
    ReasoningTier,
    PSPType,
)
from aria.kg.schema import classify_node_layer, classify_path_layers, psp_layers_covered
from aria.retrieval.path_search import find_psp_paths
from aria.retrieval.completeness import causal_completeness_score, identify_missing_layers
from aria.visualization.trace_viz import plot_causal_trace, plot_tier_comparison

## 2. Building a Demo KG

We need a knowledge graph for the engine. If the demo JSON file is not available, we build a tiny KG programmatically.

In [ ]:
from aria.types import PSPRelationship, PSPType

# Build a tiny KG (same as Tutorial 01)
kg = nx.DiGraph()

# Processing -> Structure edges
kg.add_edge(
    "CVD temperature 750C", "improved crystallinity",
    mechanism="High CVD temperature promotes ordered MoS2 crystal growth",
    affected_property="crystallinity", confidence=0.9,
    psp_type=PSPType.PROCESSING_TO_STRUCTURE.value, relation="increases",
)
kg.add_edge(
    "CVD temperature 750C", "reduced defect density",
    mechanism="Elevated temperature anneals sulfur vacancies",
    affected_property="defect density", confidence=0.85,
    psp_type=PSPType.PROCESSING_TO_STRUCTURE.value, relation="decreases",
)
kg.add_edge(
    "doping concentration Nb", "doping_level",
    mechanism="Nb substitution at Mo sites introduces carriers",
    affected_property="doping_level", confidence=0.92,
    psp_type=PSPType.PROCESSING_TO_STRUCTURE.value, relation="increases",
)
kg.add_edge(
    "CVD temperature 750C", "grain growth",
    mechanism="Higher temperature increases grain size",
    affected_property="grain_size", confidence=0.8,
    psp_type=PSPType.PROCESSING_TO_STRUCTURE.value, relation="increases",
)

# Structure -> Structure edges
kg.add_edge(
    "grain growth", "improved crystallinity",
    mechanism="Larger grains reduce grain boundary scattering",
    affected_property="crystallinity", confidence=0.75,
    psp_type=PSPType.STRUCTURE_TO_STRUCTURE.value, relation="increases",
)

# Structure -> Property edges
kg.add_edge(
    "improved crystallinity", "higher carrier mobility",
    mechanism="Ordered crystal lattice enables efficient carrier transport",
    affected_property="carrier mobility", confidence=0.88,
    psp_type=PSPType.STRUCTURE_TO_PROPERTY.value, relation="increases",
)
kg.add_edge(
    "reduced defect density", "higher carrier mobility",
    mechanism="Fewer scattering centres boost mobility",
    affected_property="carrier mobility", confidence=0.82,
    psp_type=PSPType.STRUCTURE_TO_PROPERTY.value, relation="increases",
)

print(f"Demo KG: {kg.number_of_nodes()} nodes, {kg.number_of_edges()} edges")

## 3. Understanding ARIA's Operating Modes

ARIA supports five operating modes, each representing a different strategy for combining LLM reasoning with KG evidence:

| Mode | Description | KG Used? | Literature? | CoT? |
|------|-------------|----------|-------------|------|
| `baseline` | Pure LLM, no KG | No | No | No |
| `naive_kg` | Simple KG + LLM concatenation | Yes | No | No |
| `aria` | 3-tier causal cascade | Yes | No | No |
| `aria_search` | 3-tier + literature search | Yes | Yes | No |
| `aria_full` | 3-tier + literature + CoT transparency | Yes | Yes | Yes |

The **baseline** mode uses only the LLM's parametric knowledge. The **naive_kg** mode simply concatenates all relevant KG paths into the prompt without checking causal completeness. The **aria** mode implements the full 3-tier cascade with evidence gating.

## 4. Initializing ARIAEngine

The `ARIAEngine` class is the unified entry point for all modes. It requires a knowledge graph (except in baseline mode), an LLM model name, and the operating mode.

**Note:** Running the engine requires an active Ollama server with the specified model. The cells below that call the LLM will only work if Ollama is running. We also demonstrate the path-search and routing logic that happens *before* the LLM is called, which can be run without Ollama.

In [ ]:
# Path search and routing can be done without an LLM
from aria.retrieval.path_search import find_psp_paths, extract_mechanisms
from aria.retrieval.similarity import NodeMatcher
from aria.reasoning.router import ReasoningRouter, RoutingDecision

# Pre-compute node embeddings for Tier-2 similarity matching
matcher = NodeMatcher(kg, model_name="all-MiniLM-L6-v2")
matcher.precompute()

# Create the router
router = ReasoningRouter(similarity_threshold=0.5)

print("Router and matcher initialized.")
print(f"Node embeddings computed for {len(matcher._node_list)} nodes.")

In [ ]:
# Demonstrate routing for a forward prediction query
synthesis_inputs = {
    "material": "MoS2",
    "method": "CVD",
    "temperature": "750C",
}

decision = router.route_forward(synthesis_inputs, kg, matcher)

print("Routing Decision for Forward Prediction:")
print("=" * 50)
print(f"  Tier:           {decision.tier.name} (Tier {decision.tier.value})")
print(f"  Paths found:    {len(decision.paths)}")
for i, path in enumerate(decision.paths[:5], 1):
    print(f"    Path {i}: {path}")
print(f"  Mechanisms:     {len(decision.mechanisms)}")
for i, mech in enumerate(decision.mechanisms[:3], 1):
    print(f"    Mechanism {i}: {mech[:80]}")
if decision.similar_node:
    print(f"  Similar node:   {decision.similar_node}")
    print(f"  Similarity:     {decision.similarity:.3f}")

In [ ]:
# Now try a query that would NOT match any KG nodes (triggers Tier 3)
unknown_inputs = {
    "material": "novel_2d_material",
    "method": "pulsed_laser_deposition",
    "temperature": "1200C",
}

decision_unknown = router.route_forward(unknown_inputs, kg, matcher)

print("Routing Decision for Unknown Material:")
print("=" * 50)
print(f"  Tier:           {decision_unknown.tier.name} (Tier {decision_unknown.tier.value})")
print(f"  Paths found:    {len(decision_unknown.paths)}")
print(f"  Similar node:   {decision_unknown.similar_node}")
print(f"  Similarity:     {decision_unknown.similarity:.3f}")
print()
print("This query falls to Tier 3 (Fallback) because:")
print("  1. No exact keyword match found in KG nodes")
print("  2. No semantically similar node exceeds the threshold")

## 5. Running Forward Predictions (with LLM)

The cells below require a running Ollama server. If Ollama is not available, they will raise an error. You can skip to Section 6 for the analysis of `ARIAResult` structure.

In [ ]:
# Initialize ARIAEngine in different modes
# NOTE: This requires Ollama to be running with the specified model
try:
    from aria import ARIAEngine
    
    # Initialize in aria mode (the default, recommended mode)
    engine_aria = ARIAEngine(kg=kg, model="qwen2:7b", mode="aria")
    print(f"ARIAEngine initialized in 'aria' mode")
    print(f"  KG nodes: {kg.number_of_nodes()}")
    print(f"  Model: qwen2:7b")
    print(f"  Ready for inference")
except Exception as e:
    print(f"Could not initialize engine (Ollama may not be running): {e}")
    engine_aria = None

In [ ]:
# Run a forward prediction
if engine_aria is not None:
    result = engine_aria.forward_predict(
        material="MoS2",
        processing={"temperature": "750C", "method": "CVD"},
        target_property="carrier mobility",
    )
    print("Forward Prediction Result:")
    print(f"  Answer:          {result.answer}")
    print(f"  Tier:            {result.tier.name} (Tier {result.tier.value})")
    print(f"  Confidence:      {result.confidence:.2f}")
    print(f"  Reasoning type:  {result.reasoning_type}")
    print(f"  KG paths used:   {result.kg_paths_used}")
    print(f"  Latency:         {result.latency_ms:.1f} ms")
else:
    print("Skipping prediction (Ollama not available)")
    print("The result structure would look like:")
    print(json.dumps(ARIAResult(
        answer={"carrier_type": "n-type", "mobility": "50 cm2/Vs"},
        tier=ReasoningTier.DIRECT,
        confidence=0.85,
        reasoning_type="direct_path",
        causal_trace=[CausalTraceStep(
            processing="CVD temperature 750C",
            structure="improved crystallinity",
            property_="higher carrier mobility",
            evidence_text="High CVD temperature promotes ordered crystal growth",
            confidence=0.9,
        )],
        missing_evidence=[],
        kg_paths_used=2,
        kg_paths=["CVD temperature 750C -> improved crystallinity -> higher carrier mobility"],
        mode="aria",
        model="qwen2:7b",
        latency_ms=1234.5,
    ).to_dict(), indent=2))

## 6. Inspecting ARIAResult

Every ARIA mode returns the same `ARIAResult` data structure, which contains:

- **answer**: The prediction (properties for forward, synthesis conditions for inverse)
- **tier**: Which reasoning tier was activated (1=Direct, 2=Analogical, 3=Fallback)
- **confidence**: 0-1 confidence score
- **reasoning_type**: How the prediction was made
- **causal_trace**: List of `CausalTraceStep` objects showing the PSP chain
- **missing_evidence**: PSP layers not covered by the evidence
- **kg_paths_used**: Number of KG paths used
- **kg_paths**: The actual path strings
- **literature_papers**: Papers found (aria_search and aria_full only)
- **chain_of_thought**: Detailed reasoning steps (aria_full only)
- **mode** and **model**: Configuration metadata

In [ ]:
# Demonstrate the ARIAResult structure with a synthetic example
example_result = ARIAResult(
    answer={"carrier_type": "n-type", "mobility": "50 cm2/Vs"},
    tier=ReasoningTier.DIRECT,
    confidence=0.85,
    reasoning_type="direct_path",
    causal_trace=[
        CausalTraceStep(
            processing="CVD temperature 750C",
            structure="improved crystallinity",
            property_="higher carrier mobility",
            evidence_text="High CVD temperature promotes ordered crystal growth",
            confidence=0.9,
        ),
        CausalTraceStep(
            processing="CVD temperature 750C",
            structure="reduced defect density",
            property_="higher carrier mobility",
            evidence_text="Elevated temperature anneals sulfur vacancies",
            confidence=0.85,
        ),
    ],
    missing_evidence=[],
    kg_paths_used=2,
    kg_paths=[
        "CVD temperature 750C -> improved crystallinity -> higher carrier mobility",
        "CVD temperature 750C -> reduced defect density -> higher carrier mobility",
    ],
    mode="aria",
    model="qwen2:7b",
    latency_ms=1234.5,
)

print("ARIAResult Structure:")
print("=" * 60)
print(f"  answer:            {example_result.answer}")
print(f"  tier:              {example_result.tier.name} ({example_result.tier.value})")
print(f"  confidence:        {example_result.confidence}")
print(f"  reasoning_type:    {example_result.reasoning_type}")
print(f"  kg_paths_used:     {example_result.kg_paths_used}")
print(f"  kg_paths:          {example_result.kg_paths}")
print(f"  missing_evidence:  {example_result.missing_evidence}")
print(f"  mode:              {example_result.mode}")
print(f"  model:             {example_result.model}")
print(f"  latency_ms:        {example_result.latency_ms}")
print()
print("Causal Trace Steps:")
for i, step in enumerate(example_result.causal_trace, 1):
    print(f"  Step {i}:")
    print(f"    Processing: {step.processing}")
    print(f"    Structure:  {step.structure}")
    print(f"    Property:   {step.property_}")
    print(f"    Evidence:   {step.evidence_text}")
    print(f"    Confidence: {step.confidence}")

## 7. Comparing Modes

Let us compare what happens conceptually in each mode for the same query. The key difference is how evidence is used:

- **baseline**: No KG evidence. The LLM relies entirely on parametric knowledge.
- **naive_kg**: All matching KG paths are concatenated into the prompt, regardless of causal completeness. This can include irrelevant or incomplete evidence.
- **aria**: Evidence is gated by the 3-tier cascade. Only causally complete paths activate Tier 1. Incomplete evidence triggers Tier 2 (analogical) or Tier 3 (fallback).

In [ ]:
# Show what each mode would do differently for the same query
query = {"material": "MoS2", "method": "CVD", "temperature": "750C"}

# Find paths that would be used
paths = find_psp_paths(kg, ["temperature", "CVD", "750C"], ["mobility"], max_hops=4)
mechanisms = extract_mechanisms(kg, paths)

print("Query:", query)
print(f"\nKG paths found: {len(paths)}")
for p in paths:
    layers = psp_layers_covered(p, kg)
    print(f"  {' -> '.join(p)}")
    print(f"    Layers: {layers}")
print(f"\nMechanisms: {len(mechanisms)}")
for m in mechanisms:
    print(f"  {m['source']} -> {m['target']}: {m['mechanism'][:60]}...")

print("\n" + "=" * 60)
print("Mode Behavior Comparison:")
print("=" * 60)
print()
print("baseline mode:")
print("  - No KG evidence used")
print("  - LLM relies on parametric knowledge only")
print("  - Tier: ALWAYS Tier 3 (Fallback)")
print("  - Risk: May hallucinate or give outdated information")
print()
print("naive_kg mode:")
print("  - ALL matching paths dumped into prompt")
print("  - No causal completeness check")
print("  - Risk: Incomplete or irrelevant evidence can mislead the LLM")
print("  - This is the naive RAG approach that ARIA improves upon")
print()
print("aria mode:")
print("  - Tier 1 if complete PSP path found")
print("  - Tier 2 if similar material found")
print("  - Tier 3 if no KG match")
print("  - Evidence is GATED on causal completeness")
print("  - This is the key innovation: retrieval without completeness can hurt")

## 8. Visualizing Causal Traces

ARIA provides visualization functions that render causal traces as PSP chain diagrams with layer-colored nodes.

In [ ]:
# Visualize the causal trace
plot_causal_trace(example_result, output_path="causal_trace.png")
print("Causal trace visualization saved to causal_trace.png")
print()
print("The visualization shows:")
print("  - Red nodes: Processing layer (synthesis conditions)")
print("  - Blue nodes: Structure layer (material structure)")
print("  - Green nodes: Property layer (measured properties)")
print("  - Edge labels: Confidence scores")
print("  - Title: Tier and overall confidence")

In [ ]:
# Create synthetic results for different modes to demonstrate tier comparison
baseline_result = ARIAResult(
    answer={"carrier_type": "n-type", "mobility": "30-60 cm2/Vs"},
    tier=ReasoningTier.FALLBACK,
    confidence=0.5,
    reasoning_type="baseline_llm",
    causal_trace=[],
    kg_paths_used=0,
    mode="baseline",
    model="qwen2:7b",
)

naive_kg_result = ARIAResult(
    answer={"carrier_type": "n-type", "mobility": "40-70 cm2/Vs"},
    tier=ReasoningTier.FALLBACK,
    confidence=0.6,
    reasoning_type="naive_kg",
    causal_trace=[],
    kg_paths_used=len(paths),
    mode="naive_kg",
    model="qwen2:7b",
)

aria_result = ARIAResult(
    answer={"carrier_type": "n-type", "mobility": "50 cm2/Vs"},
    tier=ReasoningTier.DIRECT,
    confidence=0.85,
    reasoning_type="direct_path",
    causal_trace=example_result.causal_trace,
    kg_paths_used=2,
    mode="aria",
    model="qwen2:7b",
)

plot_tier_comparison(
    [baseline_result, naive_kg_result, aria_result],
    output_path="tier_comparison.png",
)
print("Tier comparison saved to tier_comparison.png")

## 9. Understanding Tier Routing in Detail

The router examines the query and KG to decide which tier to use. Let us trace through the logic step by step.

In [ ]:
# Detailed routing analysis for different query types
test_queries = [
    {"material": "MoS2", "method": "CVD", "temperature": "750C"},  # Direct match
    {"material": "WS2", "method": "CVD", "temperature": "800C"},  # Similar material
    {"material": "perovskite", "method": "sol-gel", "temperature": "500C"},  # No match
]

print("Routing Analysis:")
print("=" * 70)

for i, query in enumerate(test_queries, 1):
    decision = router.route_forward(query, kg, matcher)
    
    # Compute causal completeness for any paths found
    paths = find_psp_paths(
        kg,
        start_keywords=[str(v) for v in query.values()],
        end_keywords=["mobility", "conductivity", "property"],
        max_hops=4,
    )
    
    completeness = causal_completeness_score(
        kg, paths, " ".join(str(v) for v in query.values())
    ) if paths else 0.0
    
    print(f"\nQuery {i}: {query}")
    print(f"  Routed to:       Tier {decision.tier.value} ({decision.tier.name})")
    print(f"  Paths found:     {len(decision.paths)}")
    print(f"  Completeness:    {completeness:.2f}")
    if decision.similar_node:
        print(f"  Similar node:    {decision.similar_node} (sim={decision.similarity:.3f})")
    
    tier_explanation = {
        ReasoningTier.DIRECT: "Exact PSP path found in KG -> high confidence",
        ReasoningTier.ANALOGICAL: "Similar material found via embeddings -> moderate confidence",
        ReasoningTier.FALLBACK: "No KG match -> pure LLM reasoning",
    }
    print(f"  Explanation:     {tier_explanation[decision.tier]}")

## 10. Summary and Key Takeaways

In this tutorial, we:

1. **Explored ARIA's 3-tier cascade** and understood how it routes queries
2. **Inspected `ARIAResult`** -- the unified output format for all modes
3. **Compared modes** (baseline, naive_kg, aria) and understood why naive retrieval can hurt
4. **Visualized causal traces** with PSP layer coloring
5. **Traced routing decisions** step by step

### The Central Finding

> **Retrieval without causal completeness can harm reasoning.** The naive_kg mode simply concatenates all matching KG paths into the prompt. If those paths lack structural explanation (the "Structure" layer), the LLM may make unjustified leaps from Processing directly to Property. ARIA's tier system prevents this by activating evidence only when it forms a complete causal chain.

In the next tutorial, we explore inverse design -- going from target properties back to synthesis conditions.